# Reached payment, never ordered

A short case for Quant UX Con 2026 (about 7 minutes). Three funnel levels:
**A** `checkout_start` → **B** `checkout_payment` → **C** `order_success`.
Why do people who reached B never get to C?

1. **Funnel**: where people drop out.
2. **Select**: one pattern selects the people who reached B and never reached C.
3. **Crop**: keep the payment screen and the next 10 events.
4. **Cluster**: what kinds of not-ordering are there?
5. **Drill down**: one of the clusters points at an Android release.

Data: the Lumeo event log (`lumeo_events.csv.gz`, 40,000 users) from the
workshop package, copied to `data/`. Only the raw log is used, no preprocessing from the workshop notebooks.
The slides (`slides.ipynb`) run the same pipeline through `talk.py`.

In [ ]:
import os
from pathlib import Path

os.environ["RETENTIONEERING_NO_TRACK"] = "1"
import numpy as np
import pandas as pd
import retentioneering as rete

DATA = Path("data")  # lumeo_events.csv.gz, copied in from the workshop package (README.md)
pd.set_option("display.width", 200)

events = pd.read_csv(DATA / "lumeo_events.csv.gz", parse_dates=["timestamp"])
# One build label per row: platform and app version; the web has no app version.
events["build"] = np.where(
    events.platform.eq("web"), "web", events.platform.astype(str) + " " + events.app_version.astype(str)
)
stream = rete.Eventstream(
    events, schema={"path_cols": ["user_id", "session_id"], "segment_cols": ["build", "platform", "device"]}
)
A, B, C = "checkout_start", "checkout_payment", "order_success"

## 1 · The funnel shows where, not why

In [ ]:
stream.funnel(steps=[A, B, C], path_col="user_id", sidebar_open=False, height=360)

In [ ]:
funnel = stream.funnel_data(steps=[A, B, C], path_col="user_id")
pd.DataFrame(funnel["steps"])

## 2 · One pattern, one question

`A->.*->B->[^C]*->path_end`: a checkout start, later a payment screen, and no
order from that payment screen to the end of the path.

The pattern alone is not enough: `A → B → C → B → end` also matches, through the
second B. The second condition removes everyone who ever ordered.

In [ ]:
DROPOFF = f"{A}->.*->{B}->[^{C}]*->path_end"

pattern_only = stream.filter_paths({"matches_pattern": DROPOFF}, path_col="user_id")
dropoffs = stream.filter_paths(
    [{"matches_pattern": DROPOFF}, {"op": "not", "args": [{"has_event": C}]}], path_col="user_id"
)
users = lambda s: s.to_dataframe().user_id.nunique()  # noqa: E731
print("pattern only:", users(pattern_only), "· without buyers:", users(dropoffs))

## 3 · Crop the window after the payment screen

Both anchors point at the same position: B in `A → … → B`. The end anchor moves
10 events further, or to the end of the path if it is shorter.

People who reached payment in the last two weeks of the log had no time to come
back. `first_event_time` of a cropped path is the moment of the payment screen, so
the same `filter_paths` keeps only those with at least 14 days of observation.

In [ ]:
WINDOW = 10
LOG_END = events.timestamp.max().normalize() + pd.Timedelta(days=1)
MATURE_BEFORE = LOG_END - pd.Timedelta(days=14)


def crop(dropoffs, window):
    anchor = {"pattern": f"{A}->.*->{B}"}
    cropped = dropoffs.truncate_paths(
        start_anchor=anchor, end_anchor={**anchor, "offset": window}, path_col="user_id"
    )
    return cropped.filter_paths(
        {"metric": "first_event_time", "op": "<", "value": MATURE_BEFORE.timestamp()}, path_col="user_id"
    )


cropped = crop(dropoffs, WINDOW)
print("payment before", MATURE_BEFORE.date(), "·", users(cropped), "people")
paths = cropped.to_dataframe().groupby("user_id").event.agg(" → ".join)
paths.sample(8, random_state=1).to_frame("path after the payment screen")

## 4 · Six ways not to order

Features: how many times the window has a payment error, an order review, the cart,
a product page, a search and the order list, plus the window length. k-means, 6 clusters, min-max scaled. We name a cluster after its profile:
what happened at payment, and whether the person came back.

In [ ]:
COUNTED = ["payment_error", "checkout_review", "cart", "product_view", "search", "orders_view"]
FEATURES = [{"metric": "event_count", "metric_args": {"event": e}} for e in COUNTED] + [{"metric": "length"}]


def cluster(cropped):
    clustered = cropped.add_clusters(
        "dropoff", features=FEATURES, method="kmeans",
        method_args={"n_clusters": 6, "n_init": 10, "random_state": 42}, scaler="minmax", path_col="user_id",
    )
    profile = clustered.segment_overview_data("dropoff", metrics=NAMING_METRICS, path_col="user_id")
    return clustered.rename_segment_levels("dropoff", {c: cluster_name(profile[c]) for c in profile.columns})


NAMING_METRICS = [
    {"metric": "has_event", "metric_args": {"event": "payment_error"}, "agg": "mean"},
    {"metric": "has_event", "metric_args": {"event": "checkout_review"}, "agg": "mean"},
    # A new session after the payment screen starts on home or with an app open.
    {"metric": "has_any_event", "metric_args": {"events": ["home", "app_open"]}, "agg": "mean"},
]


def cluster_name(p):
    if p["has_event_payment_error_mean"] > 0.5:
        stage = "error"
    elif p["has_event_checkout_review_mean"] > 0.5:
        stage = "review"
    else:
        stage = "payment"
    return f"{stage} · " + ("back" if p["has_any_event_home_or_app_open_mean"] > 0.5 else "gone")


clustered = cluster(cropped)
OVERVIEW = NAMING_METRICS + [{"metric": "length", "agg": "mean"}] + [
    {"metric": "has_event", "metric_args": {"event": e}, "agg": "mean"}
    for e in ["cart", "product_view", "search", "orders_view"]
]
clustered.segment_overview("dropoff", metrics=OVERVIEW, path_col="user_id", sidebar_open=False, height=420)

In [ ]:
labels = clustered.to_dataframe().groupby("user_id").dropoff.first()
examples = pd.DataFrame({"cluster": labels, "path": paths})
examples.groupby("cluster").head(2).sort_values("cluster")

## 5 · The error clusters point at one build

Two clusters start with a payment error. Which builds are behind them?

In [ ]:
ERROR_CLUSTERS = [c for c in labels.unique() if c.startswith("error")]
ANDROID = "android 4.12.0"
in_android = {"metric": "in_segment", "metric_args": {"segment_name": "build", "segment_level": ANDROID}, "agg": "mean"}
share = clustered.segment_overview_data("dropoff", metrics=[in_android], path_col="user_id").T
share.columns = ["people", "share", f"{ANDROID} share"]
share.sort_values(f"{ANDROID} share", ascending=False).round(2)

Now every session with a payment screen, not only the people who never ordered:
how often does it hit a payment error, and how often does it end in an order?

In [ ]:
payment_sessions = stream.filter_paths({"has_event": B}, path_col="session_id")
by_build = payment_sessions.segment_overview_data(
    "build",
    metrics=[
        {"metric": "has_event", "metric_args": {"event": "payment_error"}, "agg": "mean"},
        {"metric": "has_event", "metric_args": {"event": C}, "agg": "mean"},
    ],
    path_col="session_id",
)
by_build.T.round(3).sort_values("has_event_payment_error_mean", ascending=False)

In [ ]:
payment_sessions.step_matrix(
    anchor=B, max_steps=3, diff=("build", ANDROID, "<REST>"), path_col="session_id",
    sidebar_open=False, height=480,
)

## Appendix · the clusters do not depend on the window

The same six clusters come out for windows of 5, 10 and 20 events.

In [ ]:
rows = {}
for window in [5, 10, 20]:
    lab = cluster(crop(dropoffs, window)).to_dataframe().groupby("user_id").dropoff.first()
    rows[window] = lab.value_counts()
pd.DataFrame(rows).rename_axis("cluster").fillna(0).astype(int)